# Canterax Usage Guide

This notebook walks through the main public APIs in Canterax:

- loading a mechanism with `Solution`
- setting thermodynamic state
- reading scalar and species-level properties
- switching between mass and molar bases
- running equilibrium calculations
- integrating a constant-pressure reactor with `ReactorNet`
- modeling an open reactor with mass flow controllers


## Imports

These examples use the built-in `gri30.yaml` mechanism distributed with Cantera, so they do not depend on repo-local mechanism files.

In [ ]:
import cantera as ct
import diffrax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from canterax import ReactorNet, Solution
from canterax.flow import OpenReactorNet

## Create a `Solution`

`Solution` provides a Cantera-like ideal-gas API backed by JAX-native thermodynamics and kinetics.

In [ ]:
gas = Solution("gri30.yaml")
gas.TPX = 1500.0, ct.one_atm, "CH4:1, O2:2, N2:7.52"

print("Temperature [K]:", gas.T)
print("Pressure [Pa]:", gas.P)
print("cp_mass [J/kg/K]:", gas.cp_mass)
print("enthalpy_mole [J/kmol]:", gas.enthalpy_mole)
print("viscosity [Pa*s]:", gas.viscosity)
print("thermal_conductivity:", gas.thermal_conductivity)

## Composition helpers

The object exposes both mole and mass fractions, plus a few metadata helpers that mirror common Cantera workflows.

In [ ]:
print("First five species:", gas.species_names[:5])
print("Index of CH4:", gas.species_index("CH4"))
print("Mole fractions:", gas.mole_fraction_dict(threshold=1e-6))
print("Mass fractions:", gas.mass_fraction_dict(threshold=1e-6))

## Basis-aware aliases

Aliases such as `h`, `u`, `s`, `cp`, `cv`, `v`, and `density` follow the active basis. That means the same shorthand property switches between its mass-based and molar version when you change `gas.basis`.

In [ ]:
gas = Solution("gri30.yaml")
gas.TPX = 1200.0, ct.one_atm, "H2:2, O2:1, N2:3.76"

gas.basis = "mass"
print("Mass basis:")
print("  h =", gas.h)
print("  cp =", gas.cp)
print("  v =", gas.v)

gas.basis = "molar"
print("Molar basis:")
print("  h =", gas.h)
print("  cp =", gas.cp)
print("  v =", gas.v)

## State setters

Canterax supports the common Cantera-style state tuples such as `TPX`, `HPY`, `UVX`, `SPY`, `SVX`, `TDY`, and `DPX`.

In [ ]:
gas = Solution("gri30.yaml")
gas.TPX = 1200.0, 2.0 * ct.one_atm, "CH4:1, O2:2, N2:7.52"

h, p, y = gas.HPY
gas.HPY = h, p, y

u, v, x = gas.UVX
gas.UVX = u, v, x

s, p = gas.SP
gas.SP = s, p

print("State restored successfully")
print("T =", gas.T)
print("P =", gas.P)

## Species-level thermodynamic arrays

These arrays are useful for detailed post-processing and for parity with common Cantera workflows.

In [ ]:
gas = Solution("gri30.yaml")
gas.TPX = 1400.0, ct.one_atm, "CO:1, O2:0.5, N2:1.88"

print("standard_cp_R shape:", gas.standard_cp_R.shape)
print("partial_molar_enthalpies shape:", gas.partial_molar_enthalpies.shape)
print("chemical_potentials shape:", gas.chemical_potentials.shape)
print("First three standard_cp_R values:", gas.standard_cp_R[:3])

## Equilibrium

The public `equilibrate` API currently supports `TP` and `HP` equilibrium modes.

In [ ]:
gas_tp = Solution("gri30.yaml")
gas_tp.TPX = 2000.0, ct.one_atm, "CH4:1, O2:2, N2:7.52"
gas_tp.equilibrate("TP")

print("TP equilibrium:")
print("  T =", gas_tp.T)
print("  Major species =", gas_tp.mole_fraction_dict(threshold=1e-3))

In [ ]:
gas_hp = Solution("gri30.yaml")
gas_hp.TPX = 1500.0, ct.one_atm, "CH4:1, O2:2, N2:7.52"
gas_hp.equilibrate("HP")

print("HP equilibrium:")
print("  T =", gas_hp.T)
print("  Major species =", gas_hp.mole_fraction_dict(threshold=1e-3))

## Reactor integration with `ReactorNet`

`ReactorNet` advances a constant-pressure adiabatic reactor. The state vector is `[T, Y...]`, where `Y` is the species mass-fraction vector.

`saveat` controls which solution points Diffrax returns, not the solver's internal step sizes. This is important for stiff chemistry:

- the implicit solver still chooses adaptive internal steps based on error control
- `diffrax.SaveAt(ts=...)` asks the solver to report the state at specific times, typically for plotting or analysis
- `diffrax.SaveAt(t1=True)` stores only the final state and is usually the lightest option when you do not need a full trajectory
- increasing the number of output points does not force fixed-step integration; it just requests denser saved output

Use `t1=True` for endpoint workflows and `ts=...` when you need a sampled trajectory.

In [ ]:
gas = Solution("gri30.yaml")
gas.TPX = 1200.0, ct.one_atm, "CH4:1, O2:2, N2:7.52"

T0 = gas.T
P = gas.P
Y0 = jnp.array(gas.Y)
t_end = 1e-3
t_samples = jnp.linspace(0.0, t_end, 200)

net = ReactorNet(gas.mech)
saveat = diffrax.SaveAt(ts=t_samples)
sol = net.advance(T0, P, Y0, t_end, saveat=saveat)

temperature = np.array(sol.ys[:, 0])
time_ms = np.array(sol.ts) * 1e3

print("Final temperature [K]:", temperature[-1])
print("Recorded points:", len(time_ms))

In [ ]:
# If you only need the endpoint, omit saveat and keep the default final-state output.
sol_final = net.advance(T0, P, Y0, t_end)
print("Stored times:", np.array(sol_final.ts))
print("Final temperature only [K]:", float(sol_final.ys[-1, 0]))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(time_ms, temperature, linewidth=2)
plt.xlabel("Time [ms]")
plt.ylabel("Temperature [K]")
plt.title("Canterax Constant-Pressure Reactor")
plt.grid(True)
plt.show()

## Mass flow controllers and `OpenReactorNet`

For an open constant-pressure reactor, Canterax provides `OpenReactorNet`. Its state is `[T, Y..., m]`, where the final entry is the total reactor mass.

The inlet and outlet mass flow controllers are represented through `mdot_in` and `mdot_out`:

- pass a scalar for constant mass flow rate in kg/s
- pass a length-2 array `[a, b]` for a linear law `mdot(t) = a + b t`

You also specify the inlet thermodynamic state with `Tin` and `Yin`. The model then combines chemistry, flow-driven mixing, enthalpy transport, and reactor mass balance.

In [ ]:
gas_reactor = Solution("gri30.yaml")
gas_reactor.TPX = 1400.0, ct.one_atm, "CH4:1, O2:2, N2:7.52"

gas_inlet = Solution("gri30.yaml")
gas_inlet.TPX = 1100.0, ct.one_atm, "CH4:0.5, O2:2, N2:7.52"

T0 = gas_reactor.T
P = gas_reactor.P
Y0 = jnp.array(gas_reactor.Y)
m0 = 1.0

Tin = gas_inlet.T
Yin = jnp.array(gas_inlet.Y)

t_end = 2e-4
t_samples = jnp.linspace(0.0, t_end, 150)

open_net = OpenReactorNet(gas_reactor.mech)
open_sol = open_net.advance(
    T0,
    P,
    Y0,
    m0,
    t_end,
    Tin=Tin,
    Yin=Yin,
    mdot_in=0.01,
    mdot_out=0.01,
    saveat=diffrax.SaveAt(ts=t_samples),
)

open_time_ms = np.array(open_sol.ts) * 1e3
open_temperature = np.array(open_sol.ys[:, 0])
open_mass = np.array(open_sol.ys[:, -1])

print("Final open-reactor temperature [K]:", open_temperature[-1])
print("Final reactor mass [kg]:", open_mass[-1])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(open_time_ms, open_temperature, linewidth=2)
axes[0].set_xlabel("Time [ms]")
axes[0].set_ylabel("Temperature [K]")
axes[0].set_title("Open Reactor Temperature")
axes[0].grid(True)

axes[1].plot(open_time_ms, open_mass, linewidth=2)
axes[1].set_xlabel("Time [ms]")
axes[1].set_ylabel("Mass [kg]")
axes[1].set_title("Open Reactor Mass")
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Next steps

For deeper implementation details, see:

- `docs/wiki/Reactor.md`
- `docs/wiki/Equilibrium.md`
- `docs/wiki/Thermodynamics.md`
- `docs/wiki/Modules.md`
